# 01 · Tensors & Autograd

Notes and experiments from the OpenCV University *PyTorch Bootcamp* (lessons 1–2), connected to 3Blue1Brown's *Neural Networks* series.

<a href="https://colab.research.google.com/github/Zerothlaw780/pytorch-learning/blob/main/01-tensors-autograd/01_tensors_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
torch.manual_seed(0)

## 1. One layer of 3Blue1Brown's digit network as tensors

Input: a 28×28 image flattened to 784 pixels. First hidden layer: 16 neurons.
Each **row** of `W` is one neuron's weights, so `W` has shape (16, 784).

**Prediction before running:** `(16 × 784) @ (784,) → (16,)`

In [2]:
x = torch.rand(784)          # flattened 28x28 image
W = torch.randn(16, 784)     # layer-1 weights: one row per neuron
b = torch.randn(16)          # one bias per neuron
a = torch.sigmoid(W @ x + b) # weighted sum + bias → activation
print(a.shape)

torch.Size([16])


## 2. Counting parameters: where does 13,002 come from?

784 → 16 → 16 → 10

In [3]:
layers = [784, 16, 16, 10]
total = 0
for n_in, n_out in zip(layers[:-1], layers[1:]):
    weights, biases = n_out * n_in, n_out
    total += weights + biases
    print(f"{n_in:>3} → {n_out:>2}: W {n_out}×{n_in} = {weights:>6}, b = {biases:>2}")
print("total parameters:", total)

784 → 16: W 16×784 =  12544, b = 16
 16 → 16: W 16×16 =    256, b = 16
 16 → 10: W 10×16 =    160, b = 10
total parameters: 13002


## 3. Autograd on a single neuron

*   List item
*   List item



Three different things that are easy to mix up:

| | What it is |
|---|---|
| `z = w·a + b` | the neuron's raw value |
| `out = ReLU(z)` | the activation (also a value) |
| `w.grad` | **sensitivity**: how much `out` changes if `w` changes a little |

Chain rule: ∂z/∂w = a, ∂z/∂b = 1, and ReLU's slope is 1 for z > 0 and 0 for z < 0.

**Predict first, then run.**

In [7]:
for a_val in [0.9, 0.0]:
    a = torch.tensor(a_val)
    w = torch.tensor(2.0, requires_grad=True)
    b = torch.tensor(-1.0, requires_grad=True)

    z = w * a + b          # 1. weighted sum
    out = torch.relu(z)    # 2. activation
    out.backward()         # 3. gradients via the chain rule
    print(f"a={a_val} | z={z.item():.2f} | out={out.item():.2f} | "
          f"w.grad={w.grad.item():.2f} | b.grad={b.grad.item():.2f}")

a=0.9 | z=0.80 | out=0.80 | w.grad=0.90 | b.grad=1.00
a=0.0 | z=-1.00 | out=0.00 | w.grad=0.00 | b.grad=0.00


## 4. What I learned

1- When the input activation is zero, the weights gradient vanishes: since ∂z/∂w = a, changing the weight has no effect on the output.

If the previous neuron’s activation is 0, w.grad is 0, because the weight is multiplied by that activation. No matter how much we change the weight, anything multiplied by zero stays zero.

2-